In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import joblib

In [5]:
# Load the cleaned dataset

data_path = Path(
    "../data/processed/chennai_house_price_prediction_cleaned.csv"
)

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (7109, 19)


,PRT_ID,AREA,INT_SQFT,DIST_MAINROAD,N_BEDROOM,N_BATHROOM,N_ROOM,SALE_COND,PARK_FACIL,BUILDTYPE,UTILITY_AVAIL,STREET,MZZONE,QS_ROOMS,QS_BATHROOM,QS_BEDROOM,QS_OVERALL,COMMIS,SALES_PRICE
0,P03210,Karapakkam,1004,131,1.0,1.0,3,AbNormal,Yes,Commercial,AllPub,Paved,A,4.0,3.9,4.9,4.330,144400,7600000
1,P09411,Anna Nagar,1986,26,2.0,1.0,5,AbNormal,No,Commercial,AllPub,Gravel,RH,4.9,4.2,2.5,3.765,304049,21717770
2,P01812,Adyar,909,70,1.0,1.0,3,AbNormal,Yes,Commercial,ELO,Gravel,RL,4.1,3.8,2.2,3.090,92114,13159200
3,P05346,Velachery,1855,14,3.0,2.0,5,Family,No,Others,NoSewr,Paved,I,4.7,3.9,3.6,4.010,77042,9630290
4,P06210,Karapakkam,1226,84,1.0,1.0,3,AbNormal,Yes,Others,AllPub,Gravel,C,3.0,2.5,4.1,3.290,74063,7406250


In [7]:
df.columns

Index(['PRT_ID', 'AREA', 'INT_SQFT', 'DIST_MAINROAD', 'N_BEDROOM',
       'N_BATHROOM', 'N_ROOM', 'SALE_COND', 'PARK_FACIL', 'BUILDTYPE',
       'UTILITY_AVAIL', 'STREET', 'MZZONE', 'QS_ROOMS', 'QS_BATHROOM',
       'QS_BEDROOM', 'QS_OVERALL', 'COMMIS', 'SALES_PRICE'],
      dtype='str')

In [8]:
X = df.drop(['PRT_ID', 'SALES_PRICE'], axis=1)
y = df['SALES_PRICE']

In [9]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# Identify numerical and categorical features

numerical_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:")
print(numerical_columns)

print("\nCategorical features:")
print(categorical_columns)

Numerical features:
['INT_SQFT', 'DIST_MAINROAD', 'N_BEDROOM', 'N_BATHROOM', 'N_ROOM', 'QS_ROOMS', 'QS_BATHROOM', 'QS_BEDROOM', 'QS_OVERALL', 'COMMIS']

Categorical features:
['AREA', 'SALE_COND', 'PARK_FACIL', 'BUILDTYPE', 'UTILITY_AVAIL', 'STREET', 'MZZONE']


C:\Users\dhara\AppData\Local\Temp\ipykernel_32336\926598637.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train.select_dtypes(


In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical', StandardScaler(), numerical_columns),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='drop'
)

In [15]:
# Train the baseline Linear Regression model

baseline_model = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
    ]
)

baseline_model.fit(X_train, y_train)
print('Baseline model trained successfully!!')

Baseline model trained successfully!!


In [18]:
# Generate preictions
y_pred = baseline_model.predict(X_test)

comparison_df = pd.DataFrame({
    "Actual Price": y_test.to_numpy(),
    "Predicted Price": np.round(y_pred, 2)
})

display(comparison_df.head(10))

,Actual Price,Predicted Price
0,12948200,13741255.53
1,7524750,7530334.68
2,16434500,16460627.20
3,6705750,6938453.58
4,15831550,14931318.92
5,9316900,9869265.91
6,7574830,7728731.78
7,13234330,14422970.10
8,14771930,15082932.26
9,18076450,17182715.79


In [19]:
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

baseline_results = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2 Score"],
    "Value": [mae, rmse, r2]
})

display(baseline_results)

print(f"MAE: ₹{mae:,.2f}")
print(f"RMSE: ₹{rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

,Metric,Value
0,MAE,590460.126311
1,RMSE,775996.234008
2,R2 Score,0.954164


MAE: ₹590,460.13
RMSE: ₹775,996.23
R² Score: 0.9542


In [20]:
# Save baseline evaluation results
results_path = Path("../reports/baseline_model_results.csv")

results_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

baseline_results.to_csv(
    results_path,
    index=False
)

print("Baseline results saved:", results_path)

Baseline results saved: ..\reports\baseline_model_results.csv


In [21]:
# Saving the baseline model
model_path = Path(
    "../models/baseline_model.joblib"
)

# Create the models directory if it doesn't exist
model_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Save the complete fitted pipeline
joblib.dump(
    baseline_model,
    model_path
)

print("Baseline model saved successfully!")
print("Location:", model_path.resolve())

Baseline model saved successfully!
Location: C:\Dharan\Level 1 ML Projects\Chennai House Price Prediction\models\baseline_model.joblib
